# PCW Lesson 8: Gradients - Multivariate Derivatives

**Student**: Katia Gwaneza Nkurunziza  
**Date**: Session 8, Fall 2026  
**Topics**: Computational graphs, backpropagation, chain rule, Jacobians, automatic differentiation (JAX)

Gradients are fundamental to modern machine learning because most models don't have a closed-form, analytical solution for the optimal parameters. Instead, we compute gradients repeatedly and take small steps in the negative gradient direction, gradually reducing the loss. This session dives into how those gradients are actually computed: backpropagation, powered entirely by the chain rule.

## Readings

See `study_materials/session_8_study_notes.md` for deeper notes.

1. **CS231n. *Backpropagation, Intuitions*.** Deep dive into backpropagation via computational graphs and the chain rule. Read up to "Modularity: Sigmoid example" before attempting Q2A below.
2. **Smilkov & Carter. *A Neural Network Playground*.** Interactive toy neural network in the browser. We'll build one on the circles dataset from scratch in class.
3. **[Review] Murphy, K. (2022), sections 13.3.1, 13.3.2** (from Session 7) **and 13.3.4 (new)**. Ask: how do backpropagation and reverse-mode autodiff relate? How does a neural network's computation graph resemble the one solved in this PCW?
4. **[Optional] Shafkat, I. (2021). *Intro to Machine Learning with JAX*.** Not required background; this session only uses JAX to compute gradients.
5. **[Optional] RitvikMath (2021). *Jacobians*.**

## Question 1 of 3: Basic Prep - Math Exercise: Sigmoid Example

### LLM Interview

I asked an LLM to write a 3-question short-answer quiz on gradient descent and backpropagation, based on the CS231n reading, with at least one question specifically about Jacobians vs. neural network gradients. I answered each myself first, using the reading, then asked for feedback.

**Q1: What is gradient descent, and why do we need it instead of solving directly for the optimal parameters?**

My answer: Gradient descent is an iterative optimization method: compute the gradient of a loss function with respect to the parameters, then take a small step in the negative gradient direction (the direction of steepest decrease), and repeat. We need it because most modern models (neural networks especially) don't have a closed-form solution, the loss function is too complex, high-dimensional, and non-linear to solve algebraically like the normal equation in linear regression. Gradient descent trades an exact one-shot answer for an iterative, approximate one that still reliably converges.

**Q2: Why is the chain rule central to backpropagation, and what makes it "efficient"?**

My answer: A neural network is a composition of many functions (layer after layer). The chain rule lets you compute the derivative of that whole composition by multiplying together the derivatives of each individual step, without ever needing to write out one giant, unwieldy formula for the whole network. Backpropagation applies the chain rule going backward through the computation graph, reusing each node's local derivative and the gradient flowing in from downstream. This is efficient because each node only ever needs its own local derivative and the gradient handed to it, exactly the "strictly local process" the PCW's hint mentions, no node needs to know anything about the rest of the graph.

**Q3 (the required Jacobian question): In a neural network we often compute the gradient on the activation of a single unit, and the Jacobian with respect to an entire layer of weights. Why?**

My answer: A **gradient** is defined for a scalar-valued function: one number in, many numbers out, one partial derivative per input. A **Jacobian** is the generalization for a vector-valued function: many numbers in, many numbers out, a full matrix of partial derivatives (every output with respect to every input).

In a network, the final loss is a single scalar number, so the derivative of the loss with respect to any single weight is genuinely a gradient (a scalar). But an intermediate layer's output is a whole vector (say, 128 activations), and its weights form a whole matrix. The relationship between one layer's input vector and its output vector is a vector-to-vector function, so its derivative is a full Jacobian matrix.

Backpropagation never actually needs to materialize these full Jacobians. Because the final loss is scalar, at every step backprop only ever needs a **Jacobian-vector product**: the incoming gradient (a vector, since it's "the loss's sensitivity to this layer's output") multiplied by the local Jacobian, collapsing straight back down to another vector. This is exactly why reverse-mode autodiff is efficient: it never forms a full Jacobian matrix explicitly, it only ever propagates vectors backward, one layer at a time.

### Reflection on the LLM's Answers

The LLM's first-pass answers were correct but generic, it explained gradient descent and the chain rule accurately but glossed over Jacobians without ever defining what one actually is. Pushing back with "don't let them get away with it, define simpler terms first" (per the PCW's own advice) got a much better answer once I asked directly: "define a Jacobian before answering, then explain why backprop avoids computing one explicitly." The revised answer matched my own reasoning above almost exactly, once the LLM was forced to define terms instead of assuming familiarity.

## Q2A: Coding Exercise - Sigmoid Example (Scalar)

### Exercise 3.1: Manual Derivatives

Working backward from the rightmost value (`d_invert = 1.00`, since `df/df = 1` for any `f`), using the chain rule at each step (backprop is strictly local: each gate only needs its own local derivative and the gradient handed to it from downstream).

**Forward pass values** (given): `mul0=-2.00, mul1=6.00, sum01=4.00, sum012=1.00, neg=-1.00, exp=0.37, plus1=1.37, invert=0.73`

**Backward pass, step by step:**

```
d_invert = 1.00                                    (df/df = 1)

invert = 1/plus1  ->  d(invert)/d(plus1) = -1/plus1^2
d_plus1 = d_invert * (-1/plus1^2) = 1.00 * (-1/1.37^2) = -0.5344

plus1 = exp + 1  ->  d(plus1)/d(exp) = 1
d_exp = d_plus1 * 1 = -0.5344

exp = e^neg  ->  d(exp)/d(neg) = e^neg = exp itself
d_neg = d_exp * exp = -0.5344 * 0.3679 = -0.1966

neg = -sum012  ->  d(neg)/d(sum012) = -1
d_sum012 = d_neg * (-1) = 0.1966

sum012 = sum01 + w2  ->  both partial derivatives are 1 (addition passes gradient through unchanged)
d_w2 = d_sum012 * 1 = 0.1966
d_sum01 = d_sum012 * 1 = 0.1966

sum01 = mul0 + mul1  ->  again both derivatives are 1
d_mul0 = d_sum01 * 1 = 0.1966
d_mul1 = d_sum01 * 1 = 0.1966

mul1 = w1 * x1  ->  d(mul1)/d(w1) = x1,  d(mul1)/d(x1) = w1
d_w1 = d_mul1 * x1 = 0.1966 * (-2.00) = -0.3932
d_x1 = d_mul1 * w1 = 0.1966 * (-3.00) = -0.5898

mul0 = w0 * x0  ->  d(mul0)/d(w0) = x0,  d(mul0)/d(x0) = w0
d_w0 = d_mul0 * x0 = 0.1966 * (-1.00) = -0.1966
d_x0 = d_mul0 * w0 = 0.1966 * 2.00 = 0.3932
```

**Pattern worth noticing**: every multiply gate (`mul0`, `mul1`) swaps its two local derivatives, the derivative with respect to one input equals the *other* input's value, times the incoming gradient. Every add gate just passes the incoming gradient through unchanged to both of its inputs. That's the entire local rule set needed for this whole graph.

In [1]:
import numpy as np

# Forward pass
w0, w1, w2 = 2.00, -3.00, -3.00
x0, x1 = -1.00, -2.00

mul0 = w0 * x0    # -2.00
mul1 = w1 * x1    # 6.00
sum01 = mul0 + mul1     # 4.00
sum012 = sum01 + w2     # 1.00
neg = -sum012           # -1.00
exp = np.exp(neg)       # 0.37
plus1 = exp + 1         # 1.37
invert = 1/plus1        # 0.73

# Backward pass
# Remember *all* derivatives are with respect to the final function invert
d_invert = 1.00

# Here's one completed for us:
d_plus1 = d_invert * (-1/plus1**2)

d_exp = d_plus1 * 1
d_neg = d_exp * exp
d_sum012 = d_neg * (-1)

# Notice for gates with multiple inputs, when going backward
# they are responsible for derivatives of multiple values!
d_w2, d_sum01 = d_sum012, d_sum012
d_mul0, d_mul1 = d_sum01, d_sum01
d_w1, d_x1 = d_mul1 * x1, d_mul1 * w1
d_w0, d_x0 = d_mul0 * x0, d_mul0 * w0

print(f"Forward: mul0={mul0:.4f} mul1={mul1:.4f} sum01={sum01:.4f} sum012={sum012:.4f}")
print(f"         neg={neg:.4f} exp={exp:.4f} plus1={plus1:.4f} invert={invert:.4f}")
print(f"\nBackward: d_plus1={d_plus1:.4f} d_exp={d_exp:.4f} d_neg={d_neg:.4f} d_sum012={d_sum012:.4f}")
print(f"          d_w2={d_w2:.4f} d_sum01={d_sum01:.4f}")
print(f"          d_w1={d_w1:.4f} d_x1={d_x1:.4f}")
print(f"          d_w0={d_w0:.4f} d_x0={d_x0:.4f}")
print("\nMatches manual derivation above.")

Forward: mul0=-2.0000 mul1=6.0000 sum01=4.0000 sum012=1.0000
         neg=-1.0000 exp=0.3679 plus1=1.3679 invert=0.7311

Backward: d_plus1=-0.5344 d_exp=-0.5344 d_neg=-0.1966 d_sum012=0.1966
          d_w2=0.1966 d_sum01=0.1966
          d_w1=-0.3932 d_x1=-0.5898
          d_w0=-0.1966 d_x0=0.3932

Matches manual derivation above.


## Question 2 of 3

### Q2B: Coding Exercise - Calculus Derivation

Given `f(w, x) = w^T x = w1*x1 + w2*x2 + w3*x3`, and the worked example showing `df/dx = w`, find `df/dw`.

**By symmetry** (the dot product is symmetric in its two arguments): each term `wi*xi` has `d(f)/d(wi) = xi` (the same way each term had `d(f)/d(xi) = wi`). So:

```
df/dw = [df/dw1, df/dw2, df/dw3]^T = [x1, x2, x3]^T = x
```

**In general, for `f(w,x) = w^T x`**: `df/dx = w` and `df/dw = x`. Each variable's derivative is simply the *other* variable's value. This is the exact same "multiply gate swaps its inputs" pattern noticed in Q2A, just written in vector form instead of scalar form.

## Core Questions (3): Coding Exercise - Sigmoid Example (Vectorized)

Real-world inputs can have thousands of dimensions (MNIST alone is 784). We store these as vectors instead of individual variables. The underlying math doesn't change, only the implementation does, the multiply-then-add (`w0*x0 + w1*x1`) becomes one matrix multiply `w.T @ x`, using exactly the `df/dw = x`, `df/dx = w` result derived in Q2B.

In [2]:
# Forward pass
w = np.array([2.00, -3.00])
b = -3.00
x = np.array([-1.00, -2.00])

dot = w.T @ x       # 4.00
summed = dot + b    # 1.00
neg = -summed       # -1.00
exp = np.exp(neg)   # 0.37
plus1 = exp + 1     # 1.37
invert = 1/plus1    # 0.73

print(f"dot={dot:.4f} summed={summed:.4f} neg={neg:.4f} exp={exp:.4f} plus1={plus1:.4f} invert={invert:.4f}")

dot=4.0000 summed=1.0000 neg=-1.0000 exp=0.3679 plus1=1.3679 invert=0.7311


In [3]:
# Backward pass
d_invert = 1.00
d_plus1 = d_invert * (-1/plus1**2)
d_exp = d_plus1 * 1
d_neg = d_exp * exp
d_summed = d_neg * (-1)

d_b, d_dot = d_summed, d_summed

# Be careful, these two are vectors! Using Q2B's result: d(dot)/dw = x, d(dot)/dx = w
d_w = d_dot * x
d_x = d_dot * w

print(f"d_plus1={d_plus1:.4f} d_exp={d_exp:.4f} d_neg={d_neg:.4f} d_summed={d_summed:.4f}")
print(f"d_b={d_b:.4f} d_dot={d_dot:.4f}")
print(f"d_w={d_w}")
print(f"d_x={d_x}")
print("\nMatches the scalar version exactly: d_w0=d_w[0], d_w1=d_w[1], d_x0=d_x[0], d_x1=d_x[1] from Q2A.")

d_plus1=-0.5344 d_exp=-0.5344 d_neg=-0.1966 d_summed=0.1966
d_b=0.1966 d_dot=0.1966
d_w=[-0.19661193 -0.39322387]
d_x=[ 0.39322387 -0.5898358 ]

Matches the scalar version exactly: d_w0=d_w[0], d_w1=d_w[1], d_x0=d_x[0], d_x1=d_x[1] from Q2A.


## Preview: Auto-Magic Gradients (JAX)

Writing the backward pass by hand meant writing the code twice, once for the forward computation, once more for its gradients. This is how deep learning research worked before automatic differentiation libraries. JAX computes the backward pass for us automatically, we only ever write the forward pass.

**Note on setup**: the PCW pins an old JAX version (`0.4.13`) via a legacy Google Cloud Storage release URL. Since JAX is already installed in this project's environment (used already in Sessions 6-7) and this exercise only needs `jax.grad`, no version-specific feature, I used the already-installed version instead of the old pinned release.

In [4]:
import jax
import jax.numpy as jnp   # JAX has a version of NumPy that acts just like NumPy

# We first define a function for *just* the forward pass
def sigmoid_example(params, x):
    dot = params['w'].T @ x     # 4.00
    summed = dot + params['b']  # 1.00
    neg = -summed                # -1.00
    exp = jnp.exp(neg)           # 0.37
    plus1 = exp + 1              # 1.37
    invert = 1/plus1             # 0.73
    return invert

params = {'w': jnp.array([2.00, -3.00]), 'b': -3.00}  # parameters as a dict
x = jnp.array([-1.00, -2.00])

print("Forward pass output:", sigmoid_example(params, x), "(matches 0.73 from above)")

Forward pass output: 0.7310586 (matches 0.73 from above)


In [5]:
grad_sigmoid_example = jax.grad(sigmoid_example)
grads = grad_sigmoid_example(params, x)
print("JAX-computed gradients:", grads)
print("\nCompare to hand-derived values above: d_w=[-0.1966, -0.3932], d_b=0.1966")
print("Match confirmed.")

JAX-computed gradients: {'b': Array(0.19661197, dtype=float32, weak_type=True), 'w': Array([-0.19661197, -0.39322394], dtype=float32)}

Compare to hand-derived values above: d_w=[-0.1966, -0.3932], d_b=0.1966
Match confirmed.


All three approaches (manual scalar backprop, vectorized NumPy backward pass, and JAX's automatic differentiation) produce the exact same gradients: `d_w = [-0.1966, -0.3932]`, `d_b = 0.1966`. JAX just automates what we did by hand above, no new math, only less code.

## Extension: Coding Exercise - Training

Extend the code above to run gradient descent on the weights for 1000 iterations, finding a value of `w` (and `b`) such that the sigmoid example's output gets close to 0.

In [6]:
def train(params, x, grad_fn, step_size, n_iterations):
    history = []
    for i in range(n_iterations):
        grads = grad_fn(params, x)
        # Gradient descent: move each parameter opposite its gradient
        params = {
            'w': params['w'] - step_size * grads['w'],
            'b': params['b'] - step_size * grads['b'],
        }
        if (i + 1) % 200 == 0:
            history.append((i + 1, float(sigmoid_example(params, x))))
    return params, history

params_init = {'w': jnp.array([2.00, -3.00]), 'b': -3.00}
print(f"Initial output: {sigmoid_example(params_init, x):.6f}")

trained_params, history = train(params_init, x, grad_sigmoid_example, step_size=0.1, n_iterations=1000)

for iteration, output in history:
    print(f"Iteration {iteration}: output = {output:.6f}")

print(f"\nFinal output: {sigmoid_example(trained_params, x):.6f}")
print(f"Final w: {trained_params['w']}")
print(f"Final b: {trained_params['b']:.4f}")

Initial output: 0.731059


Iteration 200: output = 0.009239
Iteration 400: output = 0.004406
Iteration 600: output = 0.002888
Iteration 800: output = 0.002147
Iteration 1000: output = 0.001708

Final output: 0.001708
Final w: [ 3.2284188  -0.54317373]
Final b: -4.2284


**Result**: output drops from 0.731 to 0.0017 over 1000 iterations, gradient descent successfully finds weights that push the sigmoid's output close to 0. Each step moves `w` and `b` a little further in the direction that most decreases the output, exactly the mechanism from Session 6 (logistic regression MLE) and Session 7 (the perceptron), just now with JAX computing the gradient instead of a hand-derived formula.

## Question 3 of 3: Optional Upload

Optional file upload section, not required for this submission. Notebook committed to GitHub in full.

---

## Summary

- **Backpropagation is the chain rule, applied gate by gate, backward through a computation graph.** Every gate only needs its own local derivative and the gradient handed to it from downstream, a strictly local process.
- **Multiply gates swap their inputs**: `d(a*b)/da = b`, `d(a*b)/db = a`. **Add gates pass gradients through unchanged.** These two rules solve the entire scalar example.
- **Vectors generalize the same rules**: `d(w^T x)/dw = x` and `d(w^T x)/dx = w`, the vector version of the multiply-gate swap.
- **Gradients vs. Jacobians**: a gradient is for scalar-valued functions (like the final loss); a Jacobian is for vector-valued functions (like one layer's output with respect to its input). Backprop never materializes full Jacobians, it only ever propagates vectors backward (Jacobian-vector products), which is what makes it efficient.
- **JAX automates exactly this process.** `jax.grad` computes what we derived by hand, verified to match exactly, letting us define only the forward pass and get the backward pass for free.
- **Gradient descent, in practice**: repeatedly compute the gradient, step opposite it, and the output moves toward whatever target you're optimizing for, here, toward 0, over 1000 iterations.